In [2]:
import os
from pathlib import Path
from google.cloud import storage
import pandas as pd
import io

# 1. Encontrar la ruta absoluta del archivo sin importar dónde estés
base_path = Path.cwd()
# Si estás en jupyter_scratchpad, subimos un nivel. Si estás en raíz, nos quedamos.
if base_path.name == 'jupyter_scratchpad':
    key_path = base_path.parent / ".streamlit" / "service-account.json"
else:
    key_path = base_path / "observatory" / ".streamlit" / "service-account.json"

print(f"📍 Buscando credenciales en: {key_path}")
print(f"✅ ¿Existe? {key_path.exists()}")

# 2. Conexión usando la ruta absoluta encontrada
client = storage.Client.from_service_account_json(str(key_path))

# 3. Stream a memoria (sin ensuciar el disco)
BUCKET_NAME = "pienza-streamlit"
BLOB_NAME = "260424_NLP_h3_machine_clusters.csv"

bucket = client.bucket(BUCKET_NAME)
blob = bucket.blob(BLOB_NAME)
buffer = io.BytesIO()
blob.download_to_file(buffer)
buffer.seek(0)

df_h3 = pd.read_csv(buffer)
print(f"📊 CSV Cargado en memoria. Filas: {len(df_h3)}")
print(df_h3.head())

📍 Buscando credenciales en: /workspaces/pienza/observatory/.streamlit/service-account.json
✅ ¿Existe? True
📊 CSV Cargado en memoria. Filas: 0
Empty DataFrame
Columns: [h3_index, final_zone_name]
Index: []


In [4]:
import os
from pathlib import Path
from google.cloud import storage
import pandas as pd
import io

# 1. ENCONTRAR LA RAÍZ DEL PROYECTO (Donde sea que estés)
# Buscamos hacia arriba hasta encontrar la carpeta 'observatory'
def get_project_root():
    current = Path.cwd()
    while not (current / "observatory").exists() and current.parent != current:
        current = current.parent
    return current

project_root = get_project_root()
key_path = project_root / "observatory" / ".streamlit" / "service-account.json"

print(f"📍 Ruta absoluta de credenciales: {key_path}")
print(f"✅ ¿Existe el archivo?: {key_path.exists()}")

# 2. CARGA DE DATOS DESDE GCS (Parity Audit)
client = storage.Client.from_service_account_json(str(key_path))
bucket = client.bucket("pienza-streamlit")

# Vamos a probar ambos archivos: el de Paridad y el de H3 para ver qué tiene cada uno
files_to_check = [
    "260425_Pienza_Babel_Final_Parity_Audit.csv",
    "260424_NLP_h3_machine_clusters.csv"
]

for file_name in files_to_check:
    blob = bucket.blob(file_name)
    buffer = io.BytesIO()
    blob.download_to_file(buffer)
    buffer.seek(0)
    df = pd.read_csv(buffer)
    print(f"\n--- Auditoría de: {file_name} ---")
    print(f"Filas: {len(df)}")
    print(f"Columnas: {df.columns.tolist()}")
    print(f"Primeros IDs: {df.iloc[0].values[0] if len(df) > 0 else 'EMPTY'}")

📍 Ruta absoluta de credenciales: /workspaces/pienza/observatory/.streamlit/service-account.json
✅ ¿Existe el archivo?: True

--- Auditoría de: 260425_Pienza_Babel_Final_Parity_Audit.csv ---
Filas: 678
Columnas: ['offer_id', 'address', 'ml_input', 'true_id', 'true_name', 'babel_pred_id', 'babel_pred_name', 'babel_hit', 'babel_conf', 'beto_pred_id', 'beto_pred_name', 'beto_hit', 'beto_conf']
Primeros IDs: OCR01479

--- Auditoría de: 260424_NLP_h3_machine_clusters.csv ---
Filas: 0
Columnas: ['h3_index', 'final_zone_name']
Primeros IDs: EMPTY
